<a href="https://colab.research.google.com/github/Nour-Tamimi/BinXtraining/blob/main/Week7/Day5/Day5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


**From Day 4:**
 Zero-shot = pre-trained model, asked to guess your task cold, no extra training
 Fine-tuned = pre-trained model, then further trained specifically on your data

 Day 5's job is to fine-tune a transformer (e.g., DistilBERT) on 5000 examples so i get a number that's actually comparable to LSTM.

In [8]:
import numpy as np
import random, time
from datasets import Dataset,load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer

# Install required libraries
!pip uninstall -y torch torchvision -q
!pip install torch torchvision --index-url https://download.pytorch.org/whl/cu121 -q
!pip install transformers datasets accelerate -q

dataset = load_dataset("fancyzhx/ag_news")

# 1. Grab the SAME 5000-example subsample
random.seed(42)
idx = random.sample(range(len(dataset['train'])), 5000)

train_subset = dataset['train'].select(idx)
test_subset = dataset['test']

# 2. Load tokenizer + model (4 labels for AG News)
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=4)

# 3. Tokenize — use RAW text column, not your cleaned_text
def tokenize_fn(batch):
    return tokenizer(batch['text'], truncation=True, padding='max_length', max_length=128)

train_tokenized = train_subset.map(tokenize_fn, batched=True)
test_tokenized = test_subset.map(tokenize_fn, batched=True)

train_tokenized = train_tokenized.rename_column("label", "labels")
test_tokenized = test_tokenized.rename_column("label", "labels")

from transformers import DataCollatorWithPadding
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)


# 4. Metrics
from sklearn.metrics import accuracy_score, classification_report

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    acc = accuracy_score(labels, preds)
    return {"accuracy": acc}

# 5. Training setup
training_args = TrainingArguments(
    output_dir="./distilbert-agnews",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    eval_strategy="epoch",
    save_strategy="no",
    logging_steps=50,
    learning_rate=2e-5,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=test_tokenized,
    compute_metrics=compute_metrics,
    data_collator=data_collator,
)

# 6. Train
trainer.train()

# 7. Final evaluation
preds_output = trainer.predict(test_tokenized)
preds = np.argmax(preds_output.predictions, axis=1)
print(classification_report(test_subset['label'], preds))

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
fastai 2.8.8 requires torchvision>=0.11, but you have torchvision 0.2.0 which is incompatible.


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy
1,0.278506,0.309022,0.900921
2,0.202716,0.307128,0.905658
3,0.224148,0.311290,0.908289


              precision    recall  f1-score   support

           0       0.92      0.90      0.91      1900
           1       0.97      0.98      0.98      1900
           2       0.88      0.86      0.87      1900
           3       0.87      0.90      0.88      1900

    accuracy                           0.91      7600
   macro avg       0.91      0.91      0.91      7600
weighted avg       0.91      0.91      0.91      7600



## Fine-Tuned DistilBERT Results

**Final accuracy: 90.8%** — the best of all four approaches so far, beating
TF-IDF (87.76%), the from-scratch LSTM (80%), and the zero-shot transformer
(70.5%).

**Why it wins:** DistilBERT enters training already knowing general English
from its pre-training (unlike the LSTM, which starts from zero), and then
fine-tuning specializes that knowledge specifically for AG News's 4 topics
(unlike zero-shot, which never saw your data at all). It gets the best of
both worlds.

### Reading the training log

| Epoch | Training Loss | Validation Loss | Accuracy |
|-------|---------------|------------------|----------|
| 1 | 0.279 | 0.309 | 90.09% |
| 2 | 0.203 | 0.307 | 90.57% |
| 3 | 0.224 | 0.311 | 90.83% |

- **Training loss** dropped then ticked back up slightly (epoch 2→3) — a
  mild sign the model is starting to memorize training examples rather than
  learn general patterns.
- **Validation loss** barely moves (0.309 → 0.311) across all 3 epochs —
  it's essentially flat, meaning the model isn't overfitting badly, but it's
  also not gaining much from epoch 3.
- **Accuracy** still nudges upward each epoch, so epoch 3 is still the best
  checkpoint here — but the tiny gains (90.09% → 90.83%) suggest the model
  has mostly converged by epoch 2. Early stopping around epoch 2–3 would be
  reasonable if you retrain.

### Per-class read

Class 2 (Business) is still the weakest (f1 = 0.87), consistent with every
other model you've tried — it's a genuinely harder class to separate from
Sci/Tech, not a flaw specific to any one representation.